# Response Time + Reviews Analysis — All Hotels, Full History
**Input:**
- `2026_04_17_messages.xls` — **3 sheets** (Excel row limit = 65535, данные разбиты)
- `2026_04_17_reviews.xls`

**Output:**
- `booking_response.parquet` — per-booking reply stats (thread-based)
- `daily_response.parquet` — daily median reply time per hotel (для MrBeast-графика)
- `reviews_detail.parquet` — каждый отзыв с обогащённой инфой
- `reviews_monthly.parquet` — месячная агрегация отзывов (для графика тренда)
- `hotel_summary_v2.parquet` — сводка по отелям (ответы + отзывы + время)

---
### Как считается время ответа
1. Разбиваем переписку на треды по паузам > 4ч
2. Для каждого треда: время от первого гост. сообщения до первого ответа admin
3. Ответ позже 24ч → `NaN`
4. По брони: медиана по всем тредам

In [29]:
# ── Cell 0: Setup ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('/home/nikita/code/PlatoIsDead/notebook_prototype/sentiment_analysis/data')
MSG_PATH = DATA_DIR / '2026_04_17_messages.xls'
REV_PATH = DATA_DIR / '2026_04_17_reviews.xls'

HOTEL_NAMES    = {1: 'БС74', 2: 'МК16', 3: 'Дубай', 4: 'М73', 5: 'О-44'}
GAP_HOURS      = 4    # пауза > 4ч = новый тред
REPLY_WINDOW_H = 24   # ответ позже 24ч не считается ответом на тред

print('Ready.')

Ready.


In [30]:
# ── Cell 1: Load all 3 sheets & concatenate ───────────────────────────────────
# Sheet 1: имеет заголовок. Sheets 2-3: без заголовка (продолжение с новой строки).
COLS = ['booking_id', 'hotel_id', 'message', 'from_admin',
        'date_add', 'is_admin_read', 'is_user_read']

xl = pd.ExcelFile(MSG_PATH, engine='xlrd')
print(f"Sheets: {xl.sheet_names}")

parts = []
for i, sheet in enumerate(xl.sheet_names):
    if i == 0:
        df = pd.read_excel(MSG_PATH, sheet_name=sheet, engine='xlrd')
    else:
        df = pd.read_excel(MSG_PATH, sheet_name=sheet, engine='xlrd',
                           header=None, names=COLS)
    print(f"  '{sheet}': {len(df):,} rows")
    parts.append(df)

msg = pd.concat(parts, ignore_index=True)
msg = msg.drop_duplicates(subset=['booking_id', 'date_add', 'message', 'from_admin'])

msg['date_add']   = pd.to_datetime(msg['date_add'])
msg['is_admin']   = msg['from_admin'].fillna(0).astype(int)
msg['message']    = msg['message'].astype(str).str.strip()
msg = msg.rename(columns={'booking_id': 'ID_booking'})
msg['hotel_name'] = msg['hotel_id'].map(HOTEL_NAMES)

# Drop blank / test / file-marker rows
SKIP = r'^(nan|тест|test|бегу|file|\s*)$'
msg = msg[~msg['message'].str.lower().str.match(SKIP)].copy()
msg = msg[msg['message'].str.len() >= 1].copy()
msg = msg.sort_values(['ID_booking', 'date_add']).reset_index(drop=True)

print()
print(f"Total rows:  {len(msg):,}")
print(f"Hotels:      {sorted(msg['hotel_id'].unique())}")
print(f"Period:      {msg['date_add'].min().date()} → {msg['date_add'].max().date()}")
print(f"Bookings:    {msg['ID_booking'].nunique():,}")
print()
print(msg.groupby('hotel_name').agg(
    messages  = ('message', 'size'),
    bookings  = ('ID_booking', 'nunique'),
    guest_msg = ('is_admin', lambda x: (x == 0).sum()),
    admin_msg = ('is_admin', lambda x: (x == 1).sum()),
))

Sheets: ['messages', 'messages(2)', 'messages(3)']
  'messages': 65,535 rows
  'messages(2)': 65,536 rows
  'messages(3)': 57,205 rows

Total rows:  183,898
Hotels:      [1, 2, 3, 4, 5]
Period:      2022-03-31 → 2026-04-17
Bookings:    30,370

            messages  bookings  guest_msg  admin_msg
hotel_name                                          
БС74           31240      8025      15484      15756
Дубай          15576      5498       4805      10771
М73             5401      1680        859       4542
МК16           16607      4845       5257      11350
О-44          115074     10322      62330      52744


In [31]:
# ── Cell 2: Thread detection ──────────────────────────────────────────────────
msg['prev_time']  = msg.groupby('ID_booking')['date_add'].shift(1)
msg['gap_hrs']    = (msg['date_add'] - msg['prev_time']).dt.total_seconds() / 3600
msg['new_thread'] = msg['prev_time'].isna() | (msg['gap_hrs'] > GAP_HOURS)
msg['thread_id']  = msg.groupby('ID_booking')['new_thread'].cumsum()

n_threads    = msg.groupby(['ID_booking', 'thread_id']).ngroups
n_bookings   = msg['ID_booking'].nunique()
thread_sizes = msg.groupby(['ID_booking', 'thread_id']).size()

print(f"Тредов:             {n_threads:,}")
print(f"Броней:             {n_bookings:,}")
print(f"Среднее на бронь:   {n_threads / n_bookings:.1f}")
print(f"Однострочные треды: {(thread_sizes == 1).sum():,}")
print(f"Многострочные:      {(thread_sizes > 1).sum():,}")

Тредов:             67,063
Броней:             30,370
Среднее на бронь:   2.2
Однострочные треды: 34,097
Многострочные:      32,966


In [32]:
# ── Cell 3: Thread reply time ─────────────────────────────────────────────────
guest_msg = msg[msg['is_admin'] == 0]
admin_msg = msg[msg['is_admin'] == 1]

thread_start = (guest_msg.groupby(['ID_booking', 'thread_id'])['date_add']
                .min().rename('thread_start').reset_index())
thread_admin = (admin_msg.groupby(['ID_booking', 'thread_id'])['date_add']
                .min().rename('admin_reply').reset_index())

threads = thread_start.merge(thread_admin, on=['ID_booking', 'thread_id'], how='left')
hotel_per_booking = msg.groupby('ID_booking')['hotel_id'].first()
threads = threads.join(hotel_per_booking, on='ID_booking')
threads['hotel_name'] = threads['hotel_id'].map(HOTEL_NAMES)

threads['reply_min'] = (threads['admin_reply'] - threads['thread_start']).dt.total_seconds() / 60
threads.loc[threads['reply_min'] <= 0, 'reply_min'] = np.nan
threads.loc[threads['reply_min'] > REPLY_WINDOW_H * 60, 'reply_min'] = np.nan
threads = threads[threads['thread_start'].notna()].copy()

print(f"Тредов с гост. сообщениями: {len(threads):,}")
print(f"Ответили в 24ч:             {threads['reply_min'].notna().sum():,} "
      f"({threads['reply_min'].notna().mean()*100:.1f}%)")
print()
print(threads.groupby('hotel_name')['reply_min'].agg(
    n_threads = 'size',
    replied   = 'count',
    median    = 'median',
    mean      = 'mean',
    p90       = lambda x: x.quantile(0.9),
).round(1))

Тредов с гост. сообщениями: 34,688
Ответили в 24ч:             23,608 (68.1%)

            n_threads  replied  median  mean    p90
hotel_name                                         
БС74             7017     4314    13.7  36.5  113.6
Дубай            2579      752    29.8  48.9  126.4
М73               455      201     9.5  26.1   60.7
МК16             2761     1185     3.3  15.2   37.2
О-44            21876    17156     4.5  15.2   39.2


In [33]:
# ── Cell 3.5: Slow / unanswered threads ─────────────────────────────────────────────
# threads where reply_min is NaN: either answered after 24h or never answered
slow = threads[threads['reply_min'].isna()].copy()

slow['actual_reply_min'] = (
    (slow['admin_reply'] - slow['thread_start']).dt.total_seconds() / 60
)
slow.loc[slow['actual_reply_min'] <= 0, 'actual_reply_min'] = np.nan

first_guest = (
    guest_msg
    .sort_values('date_add')
    .groupby(['ID_booking', 'thread_id'])['message']
    .first()
    .rename('first_msg')
    .reset_index()
)
n_guest_per_thread = (
    guest_msg.groupby(['ID_booking', 'thread_id']).size()
    .rename('n_guest_msgs').reset_index()
)
slow = slow.merge(first_guest,        on=['ID_booking', 'thread_id'], how='left')
slow = slow.merge(n_guest_per_thread, on=['ID_booking', 'thread_id'], how='left')

print(f"\u0422\u0440\u0435\u0434\u043e\u0432 \u0431\u0435\u0437 \u043e\u0442\u0432\u0435\u0442\u0430 \u0432 {REPLY_WINDOW_H}\u0447: {len(slow):,}")
print(f"  \u043e\u0442\u0432\u0435\u0442\u0438\u043b\u0438 \u043f\u043e\u0437\u0436\u0435:   {slow['actual_reply_min'].notna().sum():,}")
print(f"  \u043d\u0438\u043a\u043e\u0433\u0434\u0430:            {slow['actual_reply_min'].isna().sum():,}")
print()
print(slow.groupby('hotel_name').agg(
    total      = ('thread_id', 'count'),
    late_reply = ('actual_reply_min', 'count'),
    no_reply   = ('actual_reply_min', lambda x: x.isna().sum()),
    median_h   = ('actual_reply_min', lambda x: round(x.dropna().median() / 60, 1)),
))


Тредов без ответа в 24ч: 11,080
  ответили позже:   0
  никогда:            11,080

            total  late_reply  no_reply  median_h
hotel_name                                       
БС74         2703           0      2703       NaN
Дубай        1827           0      1827       NaN
М73           254           0       254       NaN
МК16         1576           0      1576       NaN
О-44         4720           0      4720       NaN


In [34]:
# ── Cell 4: Sanity check — spot-check real threads ───────────────────────────
sample = threads[threads['reply_min'].notna()].sample(5, random_state=42)

for _, row in sample.iterrows():
    sub = msg[
        (msg['ID_booking'] == row['ID_booking']) &
        (msg['thread_id']  == row['thread_id'])
    ][['date_add', 'is_admin', 'message']].sort_values('date_add')

    print(f"[{row['hotel_name']} / бронь {row['ID_booking']} / тред {row['thread_id']}]  "
          f"reply = {row['reply_min']:.1f} мин")
    for _, m in sub.iterrows():
        who = 'ADMIN' if m['is_admin'] else 'GUEST'
        print(f"  {m['date_add'].strftime('%Y-%m-%d %H:%M')}  {who:<5}  {str(m['message'])[:80]}")
    print()

[О-44 / бронь 588154 / тред 8]  reply = 7.5 мин
  2025-04-03 14:10  GUEST  Здравствуйте, проблему не решили
  2025-04-03 14:11  GUEST  И еще отвалилась эта доска
  2025-04-03 14:12  GUEST  Направьте сейчас техника, чтобы поменял лампочку, устронил засор и пределал доск
  2025-04-03 14:12  GUEST  Уже третий день прошу
  2025-04-03 14:18  ADMIN  Добрый день, передали техникам
  2025-04-03 17:12  GUEST  Исправили ?
  2025-04-03 17:33  ADMIN  Да

[БС74 / бронь 1256543 / тред 7]  reply = 97.5 мин
  2025-04-08 18:22  GUEST  Hi . Please we need to clean our room or change this room Because she's almost f
  2025-04-08 18:22  GUEST  Please
  2025-04-08 19:59  ADMIN  Hello!
We will put your room for tomorrow in the schedule for hauskeeping.
  2025-04-08 20:13  GUEST  Okay thank you
  2025-04-08 20:22  ADMIN  )))

[БС74 / бронь 1155162 / тред 3]  reply = 41.7 мин
  2023-07-22 12:23  GUEST  Скажите пожалуйста, а уборку предусмотрена у нас? 5й день уже
  2023-07-22 13:05  ADMIN  Добрый день! Да, по

In [35]:
# ── Cell 5: Per-booking aggregation ──────────────────────────────────────────
booking_first_guest = (guest_msg.groupby('ID_booking')['date_add']
                       .min().rename('first_guest_time'))

br = (threads
      .groupby(['ID_booking', 'hotel_id', 'hotel_name'])
      .agg(
          reply_time_min = ('reply_min', 'median'),
          n_threads      = ('thread_id', 'nunique'),
          n_replied      = ('reply_min', 'count'),
      )
      .reset_index()
      .join(booking_first_guest, on='ID_booking'))

br['year'] = br['first_guest_time'].dt.year
br['date'] = br['first_guest_time'].dt.date

def bucket(m):
    if pd.isna(m): return np.nan
    if m <= 5:     return '<=5m'
    if m <= 15:    return '5-15m'
    if m <= 60:    return '15-60m'
    return '>60m'
br['reply_bucket'] = br['reply_time_min'].apply(bucket)

print(f"Броней: {len(br):,}")
print()
print(br.groupby('hotel_name')['reply_time_min'].agg(
    bookings  = 'size',
    no_reply  = lambda x: x.isna().sum(),
    resp_rate = lambda x: f"{x.notna().mean()*100:.1f}%",
    median    = 'median',
    p90       = lambda x: x.quantile(0.9),
).round(1))

Броней: 12,217

            bookings  no_reply resp_rate  median    p90
hotel_name                                             
БС74            3475      1227     64.7%    13.9  106.2
Дубай           1246       801     35.7%    30.0  112.0
М73              333       174     47.7%     8.3   55.8
МК16            1783       915     48.7%     3.3   35.5
О-44            5380      1193     77.8%     4.9   32.3


In [36]:
# ── Cell 6: Daily aggregation (MrBeast chart) ────────────────────────────────
daily = (br.dropna(subset=['reply_time_min'])
         .groupby(['hotel_id', 'hotel_name', 'date'])
         .agg(
             median_reply = ('reply_time_min', 'median'),
             mean_reply   = ('reply_time_min', 'mean'),
             n_bookings   = ('ID_booking', 'nunique'),
         ).reset_index())

daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values(['hotel_id', 'date'])
daily['rolling_30d'] = (daily.groupby('hotel_id')['median_reply']
                        .transform(lambda s: s.rolling(30, min_periods=5).mean()))

print(f"Daily rows: {len(daily):,}")
print(daily.groupby('hotel_name').agg(
    days           = ('date', 'nunique'),
    date_from      = ('date', 'min'),
    date_to        = ('date', 'max'),
    median_overall = ('median_reply', 'median'),
).round(1))

Daily rows: 2,833
            days  date_from    date_to  median_overall
hotel_name                                            
БС74         875 2022-11-01 2026-04-15            18.8
Дубай        329 2023-03-28 2026-04-15            33.4
М73          147 2023-04-03 2026-04-14             9.0
МК16         558 2023-03-28 2026-04-11             3.6
О-44         924 2023-04-13 2026-04-17             5.0


In [37]:
# ── Cell 7: Reviews — detail table ────────────────────────────────────────────
# Each review enriched with: has_text, rating_bucket, month
rev = pd.read_excel(REV_PATH, engine='xlrd')
rev['date_add']   = pd.to_datetime(rev['date_add'])
rev['hotel_name'] = rev['hotel_id'].map(HOTEL_NAMES)
rev['comment']    = rev['comment'].astype(str).replace({'nan': ''})
rev['has_text']   = rev['comment'].str.strip().str.len() > 0
rev['month']      = rev['date_add'].dt.to_period('M').dt.to_timestamp()
rev['year']       = rev['date_add'].dt.year

def rating_bucket(r):
    if pd.isna(r):  return np.nan
    if r <= 2:      return 'low'       # 1-2
    if r == 3:      return 'neutral'   # 3
    return 'high'                       # 4-5
rev['rating_bucket'] = rev['kaizen_rating'].apply(rating_bucket)

# Merge with booking response stats — does fast response = better rating?
rev_with_response = rev.merge(
    br[['ID_booking', 'reply_time_min', 'n_threads']].rename(
        columns={'ID_booking': 'booking_id'}),
    on='booking_id', how='left'
)

print(f"Всего отзывов:       {len(rev):,}")
print(f"С текстом:           {rev['has_text'].sum():,} ({rev['has_text'].mean()*100:.1f}%)")
print(f"Связано с перепиской: {rev_with_response['reply_time_min'].notna().sum():,}")
print()
print(rev.groupby('hotel_name').agg(
    n          = ('kaizen_rating', 'size'),
    avg_rating = ('kaizen_rating', 'mean'),
    with_text  = ('has_text', 'sum'),
    low        = ('rating_bucket', lambda x: (x == 'low').sum()),
    neutral    = ('rating_bucket', lambda x: (x == 'neutral').sum()),
    high       = ('rating_bucket', lambda x: (x == 'high').sum()),
).round(2))

Всего отзывов:       702
С текстом:           185 (26.4%)
Связано с перепиской: 249

              n  avg_rating  with_text  low  neutral  high
hotel_name                                                
БС74        351        4.52         83   29       12   310
Дубай        68        4.46         32    8        1    59
М73         151        4.62         28   12        3   136
МК16        132        4.71         42    5        4   123


In [38]:
# ── Cell 8: Reviews — monthly aggregation ─────────────────────────────────────
# Для графика тренда рейтинга по месяцам
reviews_monthly = (rev
    .groupby(['hotel_id', 'hotel_name', 'month'])
    .agg(
        n_reviews    = ('kaizen_rating', 'size'),
        avg_rating   = ('kaizen_rating', 'mean'),
        n_low        = ('rating_bucket', lambda x: (x == 'low').sum()),
        n_high       = ('rating_bucket', lambda x: (x == 'high').sum()),
    )
    .round(2)
    .reset_index())

print(f"Monthly rows: {len(reviews_monthly):,}")
print(reviews_monthly.groupby('hotel_name').agg(
    months         = ('month', 'nunique'),
    avg_overall    = ('avg_rating', 'mean'),
    min_month_avg  = ('avg_rating', 'min'),
).round(2))

Monthly rows: 70
            months  avg_overall  min_month_avg
hotel_name                                    
БС74            20         3.94            1.0
Дубай           18         4.28            1.0
М73             12         4.37            3.0
МК16            20         4.30            1.0


In [39]:
# ── Cell 9: Hotel summary v2 (reply + rating combined) ───────────────────────
reply_stats = (br.groupby(['hotel_id', 'hotel_name'])['reply_time_min']
               .agg(
                   n_bookings   = 'size',
                   median_reply = 'median',
                   mean_reply   = 'mean',
                   p90_reply    = lambda x: x.quantile(0.9),
                   no_reply     = lambda x: x.isna().sum(),
               ).round(1).reset_index())
reply_stats['response_rate'] = (
    (reply_stats['n_bookings'] - reply_stats['no_reply'])
    / reply_stats['n_bookings'] * 100).round(1)

rating_stats = (rev.groupby(['hotel_id', 'hotel_name'])
                .agg(
                    n_reviews      = ('kaizen_rating', 'size'),
                    avg_rating     = ('kaizen_rating', 'mean'),
                    n_with_text    = ('has_text', 'sum'),
                    low_rating_pct = ('rating_bucket',
                                       lambda x: (x == 'low').mean() * 100),
                )
                .round(2)
                .reset_index())

hotel_summary_v2 = reply_stats.merge(
    rating_stats, on=['hotel_id', 'hotel_name'], how='left')
print(hotel_summary_v2.to_string())

   hotel_id hotel_name  n_bookings  median_reply  mean_reply  p90_reply  no_reply  response_rate  n_reviews  avg_rating  n_with_text  low_rating_pct
0         1       БС74        3475          13.9        34.8      106.2      1227           64.7      351.0        4.52         83.0            8.26
1         2       МК16        1783           3.3        13.6       35.5       915           48.7      132.0        4.71         42.0            3.79
2         3      Дубай        1246          30.0        45.4      112.0       801           35.7       68.0        4.46         32.0           11.76
3         4        М73         333           8.3        24.1       55.8       174           47.7      151.0        4.62         28.0            7.95
4         5       О-44        5380           4.9        13.1       32.3      1193           77.8        NaN         NaN          NaN             NaN


In [40]:
# ── Cell 10: Save all parquet files ──────────────────────────────────────────
outputs = {
    'booking_response.parquet':  br[[
        'ID_booking', 'hotel_id', 'hotel_name',
        'first_guest_time', 'reply_time_min', 'reply_bucket',
        'n_threads', 'n_replied', 'year', 'date'
    ]],
    'daily_response.parquet':    daily,
    'reviews_detail.parquet':    rev_with_response[[
        'booking_id', 'hotel_id', 'hotel_name', 'date_add', 'month', 'year',
        'comment', 'has_text', 'kaizen_rating', 'rating_bucket',
        'reply_time_min', 'n_threads'
    ]],
    'reviews_monthly.parquet':   reviews_monthly,
    'hotel_summary_v2.parquet':  hotel_summary_v2,
    'slow_threads.parquet':     slow[[
        'ID_booking', 'hotel_id', 'hotel_name',
        'thread_id', 'thread_start', 'admin_reply',
        'actual_reply_min', 'n_guest_msgs', 'first_msg'
    ]],
}

for fname, df in outputs.items():
    path = DATA_DIR / fname
    df.to_parquet(path, index=False)
    print(f'✓  {fname:<30} {len(df):>7,} rows  {path.stat().st_size/1024:.0f} KB')

✓  booking_response.parquet        12,217 rows  263 KB
✓  daily_response.parquet           2,833 rows  69 KB
✓  reviews_detail.parquet             702 rows  34 KB
✓  reviews_monthly.parquet             70 rows  5 KB
✓  hotel_summary_v2.parquet             5 rows  8 KB
✓  slow_threads.parquet            11,080 rows  808 KB
